In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Recurrence Network — Italy INGV Catalog (1985-2025)

Run as a script  : python ITALY_network_RN.py
Convert to notebook: python convert_to_notebook.py ITALY_network_RN.py notebooks/ITALY_network_RN.ipynb
"""

import logging
import pickle
import time
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import seaborn as sns

from src.network import build_recurrence_network
from src.metrics import estimate_gamma_mle, test_power_law
from src.assortativity import compute_assortativity, plot_assortativity, plot_knn, plot_rich_club
from src.centrality import (
    plot_centrality_correlation,
    plot_top_n_cells,
    plot_geo_top_n_interactive,
    plot_geo_centrality_overlap,
)
from src.community import (
    run_louvain,
    run_consensus_louvain,
    run_spectral,
    run_infomap,
    run_hdbscan_spectral,
    run_hdbscan_geo,
    run_bigclam,
    compare_partitions,
    plot_partition_scores,
    compute_nmi_matrix,
    plot_nmi_heatmap,
    plot_community_geo,
)
from src.viz import (
    analyze_degree_distribution,
    analyze_degree_distribution_log_binning,
    plot_ccdf_with_fit,
    plot_degree_distribution_linear,
)
from src.plotutils import setup_matplotlib, configure_saves, savefig, save_plotly

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[assignment]

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
sns.set_theme(style="whitegrid")
pio.renderers.default = "notebook"

DATA_DIR    = Path("data/INGV")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / "data").mkdir(exist_ok=True)
(RESULTS_DIR / "cache").mkdir(exist_ok=True)
(RESULTS_DIR / "gephi").mkdir(exist_ok=True)

CUT_YEAR   = 1985
TARGET_CRS = "epsg:32632"  # UTM Zone 32N — Italy

# RN model parameters
RN_TARGET_DEGREE  = 20      # target ⟨k⟩; ε auto-selected by binary search
RN_FEATURE_WEIGHTS = None   # None = equal weight on (t, x, y, depth, magnitude)

SAVE_PDF: bool = True
SAVE_JPG: bool = True

setup_matplotlib()
configure_saves(SAVE_JPG, SAVE_PDF, RESULTS_DIR / "figures" / "italy" / "rn")

## Data Loading

The Italy INGV catalog covers $M \geq 1.5$ events from 1985–2025 in the bounding box
$[34°\text{N}, 48°\text{N}] \times [3°\text{E}, 22°\text{E}]$.

The Recurrence Network (RN) uses **all five catalog features simultaneously**:
origin time $t$, projected easting $x$ (km), northing $y$ (km), depth $z$ (km),
and magnitude $m$.  This is fundamentally different from all other models:
- ABE/BP/ZBZ/ETAS connect events based on the **direction of causality** (parent→offspring)
- TL/HVG connect events based on **magnitude time-series visibility**
- RN connects events based on **proximity in the full spatio-temporal-magnitude state space**

**Expected output:** ~215,000 rows sorted by origin time.

In [ ]:
print("Loading Italy earthquake catalog...")
df = pd.read_csv(DATA_DIR / "italy_earthquakes_1985_2025.csv")
df["time"] = pd.to_datetime(df["time"], utc=True)
df_net = (
    df[df["time"].dt.year >= CUT_YEAR]
    .sort_values("time")
    .reset_index(drop=True)
)
print(f"Loaded {len(df_net):,} earthquakes "
      f"({df_net['time'].dt.year.min()}–{df_net['time'].dt.year.max()})")
print(f"RAM: {df_net.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
display(pd.concat([df_net.head(3), df_net.tail(3)]))

## Feature Engineering

Each of the five catalog features is **independently normalised to $[0, 1]$**
using the catalog-wide min/max:

$$x_k^* = \frac{x_k - \min(x)}{\max(x) - \min(x)}$$

After normalisation, the five dimensions are treated symmetrically unless
`RN_FEATURE_WEIGHTS` overrides the default equal weighting.
The normalised feature matrix has shape $(N, 5)$; each row is one event.

This cell is informational — the actual normalisation happens inside
`build_recurrence_network`.  Here we visualise the feature distributions to
verify that no extreme outliers distort the recurrence threshold selection.

**Expected output:** Five histograms showing normalised feature distributions.
Time should be approximately uniform (40-year catalog at constant rate).
Magnitude should be roughly exponential (Gutenberg-Richter).
Depth should peak near the surface with a long tail (crustal seismicity).

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 5, figsize=(18, 3))
features = {
    "time (days)":     (df_net["time"] - df_net["time"].min()).dt.total_seconds() / 86400,
    "magnitude":       df_net["magnitude"],
    "depth (km)":      df_net["depth_km"],
    "longitude (°E)":  df_net["longitude"],
    "latitude (°N)":   df_net["latitude"],
}
for ax, (label, vals) in zip(axes, features.items()):
    ax.hist(vals, bins=60, color="steelblue", alpha=0.7)
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel("Count", fontsize=9)
fig.suptitle("Raw Feature Distributions — Italy Catalog", fontsize=12)
fig.tight_layout()
savefig("rn_feature_distributions_italy")
plt.show()

## Network Construction

The **Recurrence Network** (Donner et al. 2010, *New Journal of Physics*, 12, 033025)
connects events $i$ and $j$ iff their Euclidean distance in normalised feature space
is below threshold $\varepsilon$:

$$G = \bigl\{(i,j) : \|\mathbf{x}_i - \mathbf{x}_j\|_2 < \varepsilon \bigr\}$$

**Feature vector:** $\mathbf{x}_k = (t_k^*, x_k^*, y_k^*, z_k^*, m_k^*)$ — all in $[0,1]$.

**Threshold selection:** Rather than fixing $\varepsilon$ manually, the construction
auto-selects it via binary search on the KD-tree `count_neighbors` method to achieve
`RN_TARGET_DEGREE` $= 20$ average connections.  This ensures a fixed density
regardless of the specific catalog, facilitating inter-model comparison.

**Seismological interpretation:** Two events are "recurrent" if they are simultaneously
close in time, close in space (both horizontal and depth), and close in magnitude.
RN edges therefore link events that occurred under nearly identical source conditions —
seismically, these correspond to **repeating earthquakes** or aftershocks of the same
mainshock at similar epicentral distances and magnitudes.

Unlike visibility-graph models (TL, HVG) which link events based on amplitude patterns,
the RN captures genuine spatio-temporal clustering regardless of the magnitude ordering.

**Expected output:** N=215k nodes, ~2.15M edges (⟨k⟩≈20);
$\varepsilon \approx 0.01$–$0.05$ in normalised space; build time ~30–90 s.
The graph may have multiple weakly connected components if $\varepsilon$ is small.

In [ ]:
print(f"Building RN (target ⟨k⟩={RN_TARGET_DEGREE}, feature_weights={RN_FEATURE_WEIGHTS})…")
t_build = time.time()
G = build_recurrence_network(
    df_net,
    target_degree=RN_TARGET_DEGREE,
    feature_weights=RN_FEATURE_WEIGHTS,
    target_crs=TARGET_CRS,
)
print(f"Build time: {time.time() - t_build:.1f}s")
print(f"Nodes: {G.number_of_nodes():,}  Edges: {G.number_of_edges():,}")
avg_k = 2 * G.number_of_edges() / G.number_of_nodes()
print(f"Average degree: {avg_k:.2f}")
n_wcc = nx.number_connected_components(G)
gcc   = max(nx.connected_components(G), key=len)
print(f"WCC components: {n_wcc}  |  GCC size: {len(gcc):,} ({100*len(gcc)/G.number_of_nodes():.1f}%)")

## Hub Map — 2D

High-degree events in the RN are those that recur most frequently in the
spatio-temporal-magnitude neighbourhood of other events.  These correspond to
seismically active regions that produced many earthquakes under similar source
conditions — fault segments with persistent seismicity patterns, such as the
Apennine seismic belt and the Calabrian arc.

Because the RN uses **five-dimensional proximity**, hubs differ fundamentally
from TL/HVG hubs (which are the largest-magnitude mainshocks).  An RN hub need
not be large — it just needs many spatio-temporal-magnitude neighbours.
Expected hubs: regions with sustained moderate seismicity (M2–3) rather than
isolated large events.

**Expected output:** Map showing 100 hubs distributed across the Apennine belt,
Campania, and north-eastern Italy (Friuli region).  Hub magnitudes likely
M2–4 (moderate, frequently recurring), in contrast to the M5–6 TL/HVG hubs.

In [ ]:
df_nodes = pd.DataFrame([
    {
        "event_idx": n,
        "degree":    G.degree(n),
        "lat":       G.nodes[n]["lat"],
        "lon":       G.nodes[n]["lon"],
        "magnitude": G.nodes[n]["magnitude"],
    }
    for n in G.nodes()
])
df_hubs = df_nodes.nlargest(100, "degree").copy()
print(f"Top 100 hubs: degree range [{df_hubs['degree'].min():.0f}, {df_hubs['degree'].max():.0f}]")

df_hubs = df_hubs.sort_values("degree")
_deg_min = df_hubs["degree"].min()
_deg_max = df_hubs["degree"].max()
df_hubs["marker_size"] = 8 + 27 * (df_hubs["degree"] - _deg_min) / max(_deg_max - _deg_min, 1)

_IT_BOUNDS = dict(west=3, east=22, south=34, north=48)
MAP_WIDTH  = 770
MAP_HEIGHT = 700

fig = px.scatter_map(
    df_hubs, lat="lat", lon="lon",
    color="degree", size="marker_size",
    size_max=35, opacity=0.85,
    color_continuous_scale="plasma",
    map_style="carto-positron",
    hover_name="event_idx",
    hover_data={"magnitude": ":.2f", "degree": True, "marker_size": False},
    title="RN Network Hubs: Top 100 Highest-Degree Events — Italy",
)
fig.update_layout(
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    width=MAP_WIDTH, height=MAP_HEIGHT,
    map=dict(center={"lat": 41.9, "lon": 12.5}, zoom=0, bounds=_IT_BOUNDS),
)
save_plotly(fig, "rn_hub_map_2d_italy")
fig.show()

## Degree Distribution — Linear

Linear-scale histogram of node degrees.  In a recurrence network with fixed
threshold $\varepsilon$, the degree of node $i$ counts how many other events lie
within an $\varepsilon$-ball centred on $\mathbf{x}_i$ in normalised feature space.

For a uniformly random 5D point cloud the expected distribution is a Poisson
distribution (for small $\varepsilon$) with mean $\langle k \rangle = N \cdot V_5(\varepsilon)$
where $V_5(\varepsilon)$ is the 5D ball volume.  Seismic catalogs deviate because
seismicity is **clustered**: aftershock sequences create dense patches in feature space,
giving a heavy-tailed degree distribution compared to the Poisson baseline.

**Expected output:** Steeply decaying histogram with most events at $k = 5$–$30$
(near the target degree 20) and a tail extending to $k > 100$ for spatially
persistent aftershock clusters.

In [ ]:
plot_degree_distribution_linear(G, "RN Italy", max_degree=80, weighted=False)

## Degree Distribution — Log-Log

Log-log plot of $P(k)$ vs $k$.  Poisson → rapid exponential cutoff (concave on log-log).
A power-law tail ($\sim k^{-\gamma}$) would indicate scale-free clustering — that seismic
activity at different scales shares the same statistical structure in feature space.

**Expected output:** Faster-than-power-law bulk with a possibly linear upper tail.
The log-log plot should be more convex than the TL/HVG plots because the RN threshold
creates a soft cutoff at $k \approx \varepsilon \cdot N \cdot V_5(1)$.

In [ ]:
analyze_degree_distribution(G, "RN Italy")

## Degree Distribution — Log Binning

Log-binned log-log plot (Clauset et al. 2009) stabilises the sparse high-$k$ tail.
The apparent slope provides a visual pre-MLE estimate of the tail exponent.

In [ ]:
analyze_degree_distribution_log_binning(G, "RN Italy", k_min_fit=10)

## Degree Distribution — CCDF

CCDF $P(K \geq k)$ avoids binning artefacts.  Convexity on log-log confirms
a heavier-than-power-law cutoff; a straight segment at high $k$ indicates
scale-free clustering among the most active seismic regions.

In [ ]:
plot_ccdf_with_fit(G, "RN Italy", k_min_fit=10)

## MLE Gamma

Maximum-likelihood tail exponent (Clauset, Shalizi & Newman 2009):

$$\hat{\gamma} = 1 + n\left[\sum_{i=1}^{n}\ln\frac{k_i}{k_{\min}}\right]^{-1}$$

For a recurrence network on a purely random (Poisson) point cloud, the degree
distribution is approximately Poisson → exponential tail → $\hat{\gamma} \to \infty$
(no power law).  A finite $\hat{\gamma}$ with $R > 0$ (power-law preferred over
exponential) would confirm that seismic clustering in feature space produces scale-free
recurrence structure — events at many scales share the same neighbourhood statistics.

**Expected output:** $\hat{\gamma} \approx 2.5$–$5.0$; the CSN test may favour the
power-law model ($R > 0$, $p < 0.05$) because aftershock clustering creates a heavy
tail above the Poisson expectation.

In [ ]:
degs     = [d for _, d in G.degree()]
gamma_rn = estimate_gamma_mle(degs, k_min=10)
print(f"MLE γ (degree, k_min=10): {gamma_rn:.3f}")

result_rn = test_power_law(degs, k_min=10)
print(f"  CSN test: γ={result_rn['gamma']} (σ={result_rn['sigma']})  "
      f"R={result_rn['R']}  p={result_rn['p_value']}  → {result_rn['verdict']}")

## Macroscopic Metrics

**Connectivity:** With $\varepsilon$ auto-selected for $\langle k \rangle = 20$,
the RN may have multiple weakly connected components if some events are isolated in
feature space.  The giant connected component (GCC) fraction indicates how well-connected
the Italian seismicity is in this 5D representation.

**Clustering coefficient** $C$ measures the fraction of triples where all three events
are mutually recurrent.  In a random geometric graph (RGG) in $d$ dimensions,
$C_{\text{RGG}} \propto \varepsilon^d$ — higher than ER for any $d$.  For the RN,
clustering is expected to be very high ($C \approx 0.5$–$0.8$) because spatial
proximity satisfies the triangle inequality: if $i$ is close to $j$ and $j$ is close
to $k$, then $i$ is likely close to $k$.

**Average path length** $L$ in a high-clustering graph can still be short if
long-range recurrences (similar but temporally distant events) provide shortcuts.
The small-world signature $C \gg C_{\text{ER}}$ with $L \approx L_{\text{ER}}$
is expected to be very strong for the RN — stronger than any other model here —
because the geometric structure naturally produces local cliques.

*ER baselines:* $C_{\text{ER}} = \langle k \rangle / N$;
$L_{\text{ER}} = \ln N / \ln \langle k \rangle$.

**Expected output:** $C \approx 0.4$–$0.7$; $C / C_{\text{ER}} \approx 1{,}000$–$10{,}000\times$;
$L \approx 5$–$20$ hops (shorter than TL/HVG due to the high clustering creating shortcuts);
GCC fraction $> 99\%$.

In [ ]:
N_nodes = G.number_of_nodes()
M_edges = G.number_of_edges()
avg_k   = 2 * M_edges / N_nodes
print(f"Nodes: {N_nodes:,}  Edges: {M_edges:,}  ⟨k⟩ = {avg_k:.2f}")
print(f"GCC: {len(gcc):,} nodes ({100*len(gcc)/N_nodes:.1f}%)")

# Work on GCC for path-length computation
G_gcc = G.subgraph(gcc).copy()

print("Computing clustering coefficient (full graph)…")
avg_c = nx.average_clustering(G)
c_er  = avg_k / N_nodes
print(f"Avg C = {avg_c:.4f}  (ER: {c_er:.4f}  ratio: {avg_c/max(c_er,1e-12):.1f}×)")

print("Approximating avg path length on GCC (500 seeds)…")
rng      = np.random.default_rng(42)
N_gcc    = G_gcc.number_of_nodes()
seeds_apl = list(rng.choice(list(G_gcc.nodes()), size=min(500, N_gcc), replace=False))
apl_vals  = []
for s in seeds_apl:
    lengths = nx.single_source_shortest_path_length(G_gcc, s)
    apl_vals.extend(lengths.values())
avg_L = np.mean(apl_vals)
l_er  = np.log(N_nodes) / np.log(avg_k) if avg_k > 1 else float("nan")
print(f"Approx L (GCC) ≈ {avg_L:.2f}  (ER: {l_er:.2f})")
print(f"Small-world: C/C_ER = {avg_c/max(c_er,1e-12):.1f}×,  L/L_ER = {avg_L/max(l_er,1e-12):.2f}×")

## Centrality Analysis

Nine centrality measures on the RN identify seismically persistent regions —
locations that appear repeatedly in the feature-space neighbourhood of many
other events.

- **Degree:** counts recurrent feature-space neighbours; highest for regions
with sustained moderate-to-low magnitude seismicity.
- **Betweenness:** identifies events bridging otherwise disconnected recurrence
clusters — corresponding to the physical transition zones between distinct
fault systems (e.g. the transition from Apennine to Calabrian arc seismicity).
- **PageRank:** propagates influence through the recurrence graph; rewards events
connected to other high-degree recurrence hubs.
- **Eigenvector / Katz:** weight connections by the importance of neighbours,
rewarding events connected to other high-degree recurrence hubs.
- **HITS (hubs/authorities):** on the undirected RN, hubs and authorities are
symmetric, but HITS still differentiates structurally from degree in the
presence of dense recurrence clusters.
- **Harmonic centrality**: $C_H(v) = \sum_{u \neq v} 1/d(u,v)$, summing inverse
shortest-path distances. The RN can be disconnected (events in truly distinct
dynamical regimes share no $\varepsilon$-neighbours); harmonic centrality
handles this gracefully and identifies events that are globally reachable
within the recurrence structure — seismologically, the most "typical" events
that link the different active fault domains (*Bavelas 1950; Rochat 2009*).
- **Clustering coefficient**: fraction of a node's neighbour pairs that are
mutually connected ($C_i = 2t_i / k_i(k_i-1)$). In the RN, high clustering
directly measures recurrence density — events that repeatedly visit the same
compact region of the 5D feature space form tight triangles; these correspond
to persistent source zones producing seismicity under stable stress conditions
(*Marwan et al. 2009*).
- **Triangle count**: raw triangles per node. The RN is specifically designed
to have many triangles (recurrence implies mutual proximity); high counts
identify the densest recurrence clusters and confirm non-trivial topology
beyond a random graph.

Centralities are computed inline because `compute_all_centralities` expects
ABE-format ``"cx_cy_cz"`` string node IDs.

**Expected output:** Top-degree events concentrated in the most active aftershock
sequences (Amatrice 2016 sequence, L'Aquila 2009 sequence).  Betweenness may pick
up spatially isolated but temporally linking events — connecting different seismic
episodes across the catalog.

In [ ]:
print("Computing centralities for RN…")
_t0_cent = time.time()

G_nsl = G.copy()
G_nsl.remove_edges_from(nx.selfloop_edges(G_nsl))
_N    = G.number_of_nodes()

deg_cent = nx.degree_centrality(G)
print(f"  Degree done ({time.time()-_t0_cent:.1f}s)")

pr_cent = nx.pagerank(G, weight="weight")
print(f"  PageRank done ({time.time()-_t0_cent:.1f}s)")

bet_cent = nx.betweenness_centrality(G, k=min(500, _N), seed=42, normalized=True)
print(f"  Betweenness done ({time.time()-_t0_cent:.1f}s)")

try:
    eig_cent = nx.eigenvector_centrality(G, weight="weight", max_iter=500, tol=1e-6)
except nx.PowerIterationFailedConvergence:
    try:
        eig_cent = nx.eigenvector_centrality_numpy(G, weight="weight")
    except nx.AmbiguousSolution:
        eig_cent = {n: 0.0 for n in G.nodes()}
        print("  Eigenvector: skipped (disconnected graph)")
print(f"  Eigenvector done ({time.time()-_t0_cent:.1f}s)")

_max_deg    = max(d for _, d in G.degree())
_alpha_katz = 0.85 / _max_deg
try:
    katz_cent = nx.katz_centrality(
        G, alpha=_alpha_katz, weight="weight", normalized=True, max_iter=1000, tol=1e-6)
except nx.PowerIterationFailedConvergence:
    katz_cent = nx.katz_centrality_numpy(G, alpha=_alpha_katz, weight="weight")
print(f"  Katz done ({time.time()-_t0_cent:.1f}s)")

try:
    hits_hub, hits_auth = nx.hits(G_nsl, max_iter=1000, tol=1e-6)
except nx.PowerIterationFailedConvergence:
    _zeros    = {n: 0.0 for n in G.nodes()}
    hits_hub  = _zeros.copy()
    hits_auth = _zeros.copy()
print(f"  HITS done ({time.time()-_t0_cent:.1f}s total)")

harm_cent = nx.harmonic_centrality(G)
print(f"  Harmonic done ({time.time()-_t0_cent:.1f}s)")

clust_cent = nx.clustering(G, weight="weight")
print(f"  Clustering done ({time.time()-_t0_cent:.1f}s)")
tri_count = nx.triangles(G)
print(f"  Triangles done ({time.time()-_t0_cent:.1f}s)")

df_centrality = pd.DataFrame([
    {
        "cell_id":     n,
        "lat":         G.nodes[n]["lat"],
        "lon":         G.nodes[n]["lon"],
        "depth_km":    G.nodes[n]["depth_km"],
        "Degree":      deg_cent.get(n, 0.0),
        "PageRank":    pr_cent.get(n, 0.0),
        "Betweenness": bet_cent.get(n, 0.0),
        "Eigenvector": eig_cent.get(n, 0.0),
        "Katz":        katz_cent.get(n, 0.0),
        "HITS_Hub":    hits_hub.get(n, 0.0),
        "HITS_Auth":   hits_auth.get(n, 0.0),
        "Harmonic":    harm_cent.get(n, 0.0),
        "Clustering":  clust_cent.get(n, 0.0),
        "Triangles":   float(tri_count.get(n, 0)),
    }
    for n in G.nodes()
    if "lat" in G.nodes[n] and "lon" in G.nodes[n]
])

for metric, label in [
    ("Degree",      "Most Recurrent Events"),
    ("PageRank",    "Recurrence Influencers"),
    ("Betweenness", "Cluster Bridges"),
    ("HITS_Hub",    "Recurrence Hubs"),
    ("HITS_Auth",   "Recurrence Authorities"),
    ("Harmonic",    "Topological Reach (Harmonic)"),
    ("Clustering",  "Triangle Density (Clustering)"),
    ("Triangles",   "Feedback Loops (Triangles)"),
]:
    print(f"\n--- Top 5: {label} ({metric}) ---")
    display(df_centrality.nlargest(5, metric)[
        ["cell_id", "lat", "lon", "depth_km", metric]])

plot_centrality_correlation(df_centrality, "RN Italy")
plot_top_n_cells(df_centrality, top_n=10, title="RN Italy")

## Centrality Geo Maps

Geographic projection of the top-10 events per centrality metric.

RN centrality maps are expected to show a **distributed pattern** across all
major active fault systems — very different from the TL/HVG maps where a few
large mainshocks dominate.  The RN rewards persistent moderate seismicity
rather than isolated large events.

The **overlap map** identifies events with multi-metric dominance — those that
are simultaneously high-degree (many recurrent neighbours), high-betweenness
(bridge between clusters), and high-PageRank (connected to other hubs).
In the RN context, these are the most "seismically typical" locations —
areas that have produced many earthquakes under consistent source conditions
across the full 40-year catalog.

**Expected output:** More geographically spread hubs than any other model.
Likely concentration in the central Apennines (Amatrice–L'Aquila corridor),
Calabria, and Friuli.

In [ ]:
plot_geo_top_n_interactive(
    df_centrality, top_n=10,
    title="RN Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
)
plot_geo_centrality_overlap(
    df_centrality, top_n=10,
    title="RN Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
)

## Community Detection — Full Suite

Seven community-detection algorithms are applied to the undirected RN.  Modularity

$$Q = \frac{1}{2m}\sum_{ij}\left[A_{ij} - \frac{k_i k_j}{2m}\right]\delta(c_i, c_j)$$

(Newman & Girvan 2004) is the primary optimisation target for graph-based methods,
but density- and affiliation-based methods operate on complementary representations.

RN communities are **spatio-temporal-magnitude clusters**: events in the same
community occurred in a similar region, at similar depth, and with similar
magnitudes.  They correspond to distinct seismic sequences or fault segments that
produced internally homogeneous earthquakes — making the RN the only model where
community boundaries are simultaneously spatial AND magnitude-based.  Because the
RN is geometrically structured (high clustering, short paths), community boundaries
are expected to be very clean, with high modularity ($Q > 0.5$) and excellent
cross-method agreement (NMI $> 0.8$).

**Graph-based methods**
- **Louvain** (Blondel et al. 2008): greedy modularity maximisation; fast and
scalable; expected to find sharp communities due to the RN's block-like structure.
- **Consensus Louvain** (Lancichinetti & Fortunato 2012): 100-run ensemble average
over co-assignment frequencies; the most reproducible partition.
- **Spectral** (Von Luxburg 2007): Fiedler-vector bipartition applied recursively;
run on the full graph since the RN is connected for sufficiently large $\varepsilon$.
- **InfoMap** (Rosvall & Bergstrom 2008): minimises description length of a random
walk; may identify finer sub-clusters within large RN cliques.

**Density-based methods**
- **HDBSCAN-Spectral** (Campello et al. 2013): applies HDBSCAN to the
$k$-dimensional spectral embedding of the normalised graph Laplacian.
The number of communities is determined entirely by the data — no $k$
pre-specification is required.  For the RN, spectral embedding directly
reflects the 5D feature-space geometry, making HDBSCAN-Spectral a natural
double-check on the modularity-based algorithms.
- **HDBSCAN-Geographic**: same HDBSCAN algorithm applied to projected $(x, y)$
node coordinates in kilometres (EPSG:32632); no graph structure is used.
Because the RN feature space includes latitude, longitude, depth, time, and
magnitude, comparing HDBSCAN-Spectral with HDBSCAN-Geographic directly
measures how much of the detected community structure is attributable to
spatial proximity alone versus the full multivariate feature-space structure.

**Overlapping-community method**
- **BigCLAM** (Yang & Leskovec 2013): solves for an $N \times k$ affiliation
matrix $F$ via non-negative matrix factorisation; the hard partition is recovered
by $\arg\max_c F_{ic}$.  The RN is the model most likely to exhibit genuine
overlapping communities: events at the boundary of two fault segments share
similar features with neighbours on both sides and may hold legitimate affiliation
in two communities.

**Partition quality scoring**
All seven partitions are evaluated on 9 quality metrics — modularity, conductance,
coverage, Ncut, map equation, DC-SBM log-likelihood, Surprise, geographic
compactness, and depth coherence — via `compare_partitions`.

*References:* Newman & Girvan (2004) *Phys. Rev. E* 69, 026113;
Blondel et al. (2008) *J. Stat. Mech.* P10008;
Rosvall & Bergstrom (2008) *PNAS* 105, 1118;
Campello et al. (2013) *ECML-PKDD* 160–175;
Yang & Leskovec (2013) *ACM TKDD* 7(3), 1–42.

**Expected output:** 10–50 communities (more than TL/HVG because the 5D clustering
resolves finer distinctions than pure time-series models); clear geographic coherence
with communities aligning to known fault systems; high cross-method NMI ($> 0.8$)
reflecting the clean block-like structure of the RN.  HDBSCAN-Geographic NMI with
HDBSCAN-Spectral quantifies the spatial vs. multivariate contribution to structure.

In [ ]:
print("Running community detection on RN…")

print("Louvain…")
community_map = run_louvain(G, seed=42)
k_louvain     = len(set(community_map.values()))
print(f"  {k_louvain} communities")
plot_community_geo(
    G, community_map,
    title="RN Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Louvain",
)

print("Consensus Louvain (100 runs)…")
community_map_consensus = run_consensus_louvain(G, n_runs=100, seed=42)
print(f"  {len(set(community_map_consensus.values()))} communities")
plot_community_geo(
    G, community_map_consensus,
    title="RN Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Consensus Louvain",
)

print(f"Spectral (k={min(k_louvain, 8)})…")
_k_spec = min(k_louvain, 8)
community_map_spectral = run_spectral(G, k=_k_spec, seed=42)
print(f"  {len(set(community_map_spectral.values()))} communities")
plot_community_geo(
    G, community_map_spectral,
    title="RN Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Spectral",
)

print("InfoMap (undirected)…")
community_map_infomap = run_infomap(G, directed=False, seed=42)
print(f"  {len(set(community_map_infomap.values()))} communities")
plot_community_geo(
    G, community_map_infomap,
    title="RN Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="InfoMap",
)

partitions = {
    "Louvain":   community_map,
    "Consensus": community_map_consensus,
    "Spectral":  community_map_spectral,
    "InfoMap":   community_map_infomap,
}
print("Computing NMI…")
nmi_df = compute_nmi_matrix(partitions)
display(nmi_df.round(3))
plot_nmi_heatmap(nmi_df, title="RN Italy")

print("HDBSCAN-Spectral…")
community_map_hdbscan_spec = run_hdbscan_spectral(G, min_cluster_size=10, seed=42)
print(f"  {len(set(community_map_hdbscan_spec.values()))} clusters")
plot_community_geo(
    G, community_map_hdbscan_spec,
    title="RN Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=20, method_name="HDBSCAN-Spectral",
)

print("HDBSCAN-Geographic…")
community_map_hdbscan_geo = run_hdbscan_geo(G, min_cluster_size=10)
print(f"  {len(set(community_map_hdbscan_geo.values()))} clusters")
plot_community_geo(
    G, community_map_hdbscan_geo,
    title="RN Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=20, method_name="HDBSCAN-Geographic",
)

print("BigCLAM…")
F_bigclam, community_map_bigclam = run_bigclam(
    G, k=k_louvain, n_iter=100, lr=0.005, seed=42,
)
print(f"  {len(set(community_map_bigclam.values()))} non-empty communities")
plot_community_geo(
    G, community_map_bigclam,
    title="RN Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=20, method_name="BigCLAM",
)

partitions_ext = {
    **partitions,
    "HDBSCAN-Spec": community_map_hdbscan_spec,
    "HDBSCAN-Geo":  community_map_hdbscan_geo,
    "BigCLAM":      community_map_bigclam,
}
nmi_ext = compute_nmi_matrix(partitions_ext)
display(nmi_ext.round(3))
plot_nmi_heatmap(nmi_ext, title="RN Italy — all methods")

scores_df = compare_partitions(G, partitions_ext, cell_size_km=10.0)
display(scores_df.round(4))
plot_partition_scores(scores_df, title="RN Italy")

## Community Markov Flow

The Louvain partition coarse-grains the time-directed RN into a $K \times K$
flow matrix $F_{ij}$ counting directed edges from community $i$ to community $j$,
where edges are oriented by the time arrow ($i \to j$ if event $i$ precedes $j$).

$$T_{ij} = \frac{F_{ij}}{\sum_j F_{ij}}$$

In the RN context, $T_{ij}$ is the probability that the seismic catalog transitions
from a period of activity in community $i$ to activity in community $j$ between two
consecutive recurrent events.  High self-retention ($T_{ii} \approx 1$) means the
seismicity tends to stay in the same feature-space cluster — sustained aftershock
sequences.  Off-diagonal entries indicate jumps between distinct seismic episodes.

The **stationary distribution** $\boldsymbol{\pi}$ gives the long-run fraction of
catalog time spent in each feature-space community — effectively, which seismic source
zones contributed most to the 40-year catalog.

*Reference:* Rosvall, M., & Bergstrom, C. T. (2008). *PNAS*, 105(4), 1118–1123.

**Expected output:** High self-retention (0.90–0.99) reflecting persistent clustering;
the stationary distribution dominated by the 2–3 largest communities (central Apennines,
southern Italy).

In [ ]:
from src.community_flow import (
    build_community_flow_matrix,
    compute_markov_chain,
    compute_stationary_distribution,
    community_flow_stats,
    plot_flow_heatmap,
    plot_flow_entropy,
    plot_stationary_distribution,
)

# Build directed version of RN for flow computation (orient edges by time)
G_dir = nx.DiGraph()
G_dir.add_nodes_from(G.nodes(data=True))
G_dir.add_edges_from(
    (u, v) if u < v else (v, u)
    for u, v in G.edges()
)

print("Building community flow matrix (Louvain partition, time-directed RN)…")
flow_count_df = build_community_flow_matrix(G_dir, community_map)
T_markov  = compute_markov_chain(flow_count_df)
stat_dist = compute_stationary_distribution(T_markov)
df_flow   = community_flow_stats(T_markov, flow_count_df, stat_dist)

K_comm = T_markov.shape[0]
print(f"Markov chain: {K_comm} community states")
print(f"Mean self-retention:  {df_flow['self_retention'].mean():.3f}")
print(f"Mean outflow entropy: {df_flow['entropy'].mean():.3f} bits "
      f"(max = log₂({K_comm}) = {np.log2(K_comm):.3f})")
display(df_flow)

plot_flow_heatmap(T_markov, flow_count_df, title="RN Italy")
plot_flow_entropy(df_flow, K=K_comm, title="RN Italy")
plot_stationary_distribution(df_flow, title="RN Italy")

out_path = RESULTS_DIR / "data" / "italy_rn_community_flow.csv"
df_flow.to_csv(out_path, index=False)
print(f"Community flow stats saved → {out_path}")

## Assortativity

Newman (2003) assortativity on the RN tests whether recurrence links preferentially
connect events with similar scalar attributes:

$$r = \frac{\sum_{jk} jk (e_{jk} - q_j q_k)}{\sigma_q^2}$$

**Magnitude assortativity** $r$ is expected to be **strongly positive** for the RN
because the normalised magnitude $m^*$ is explicitly included in the feature vector —
events recur with each other only if their magnitudes are similar.  This is guaranteed
by construction (unlike TL/HVG where magnitude assortativity emerges from the visibility
rule) and distinguishes the RN from all other models.

**Degree assortativity** is also expected to be positive (assortative) for the RN:
events in dense aftershock clusters have many neighbours, and those neighbours tend
to also have many neighbours (they are all in the same dense patch of feature space).
This is the opposite of the disassortative hub-spoke structure of BP/ZBZ/ETAS.

*Reference:* Newman, M. E. J. (2003). Mixing patterns in networks. *Physical Review E*, 67(2), 026126.

**Expected output:** Magnitude $r \approx 0.3$–$0.7$ (strongly assortative — guaranteed
by the feature-space construction); degree $r \approx 0.1$–$0.4$ (mildly to moderately
assortative, opposite sign from BP/ZBZ).

In [ ]:
print("Computing assortativity…")
df_assort = compute_assortativity(G)
display(df_assort)
plot_assortativity(G, title="RN Italy")

print("Average nearest-neighbour degree k_nn(k):")
plot_knn(G, title="RN Italy", gamma=gamma_rn)

print("Rich-club coefficient:")
plot_rich_club(G, title="RN Italy")

## Export Results

Centrality metrics, community assignments, and the network are persisted for
downstream comparison with the other six Italy network models.

**CSV** (`italy_rn_network_metrics.csv`): one row per node; use to compare RN
degree rankings to BP out-degree and TL/HVG degree rankings.

**Pickle** (`italy_rn.pkl`): the full `nx.Graph` object with all node attributes.
Load for fast graph access in comparison notebooks.

**GEXF** (`italy_rn.gexf`): Gephi export.  Suggested layout: ForceAtlas2 with
`Prevent Overlap` enabled — the RN's high clustering creates tight cliques that
ForceAtlas2 separates clearly.  Colour by `community` (categorical) or
`magnitude` (sequential plasma).

**Expected output:** CSV with ~215,000 rows × 11 columns; pickle ~60 MB; GEXF ~250 MB.

In [ ]:
df_comm = pd.DataFrame(
    [{"cell_id": n, "community": community_map[n]} for n in community_map]
)
df_final = df_centrality.merge(
    df_comm[["cell_id", "community"]], on="cell_id", how="left"
)
out_path = RESULTS_DIR / "data" / "italy_rn_network_metrics.csv"
df_final.to_csv(out_path, index=False)
print(f"Results saved → {out_path}  ({len(df_final):,} rows)")

pkl_path = RESULTS_DIR / "cache" / "italy_rn.pkl"
with open(pkl_path, "wb") as f:
    pickle.dump(G, f)
print(f"Network cached → {pkl_path}")

gexf_path = RESULTS_DIR / "gephi" / "italy_rn.gexf"
nx.write_gexf(G, gexf_path)
print(f"Gephi export → {gexf_path}")